In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#F7F9FC',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': '#DDE3EC',
    'grid.linewidth': 0.7,
    'font.family': 'sans-serif',
})

NAVY   = '#1B3A6B'
ORANGE = '#E85D26'
GREEN  = '#2E8B57'
AMBER  = '#F4A31B'
RED    = '#C0392B'
PURPLE = '#7B2FBE'
MUTED  = '#6B7280'


In [ ]:
# Credit snapshots — each file is a point-in-time view
jan = pd.read_csv('Credit Data - 01-01-2025.csv')
mar = pd.read_csv('Credit Data - 30-03-2025.csv')
jun = pd.read_csv('Credit Data - 30-06-2025.csv')
sep = pd.read_csv('Credit Data - 30-09-2025.csv')
dec = pd.read_csv('Credit Data - 30-12-2025.csv')

# Sales and Customer Data has 4 sheets
sales_det = pd.read_excel('Sales and Customer Data.xlsx', sheet_name='Sales Details')
dob_raw   = pd.read_excel('Sales and Customer Data.xlsx', sheet_name='DOB')
inc_raw   = pd.read_excel('Sales and Customer Data.xlsx', sheet_name='Income Level')
gen_raw   = pd.read_excel('Sales and Customer Data.xlsx', sheet_name='Gender')

nps_raw = pd.read_excel('NPS Data.xlsx')

print('Snapshot sizes:')
for label, df in [('Jan',jan),('Mar',mar),('Jun',jun),('Sep',sep),('Dec',dec)]:
    print(f'  {label}: {len(df):,} rows')
print(f'NPS: {len(nps_raw):,} rows')


In [ ]:
# Add snapshot date and combine
snapshot_map = {
    'Jan 2025': (jan, pd.Timestamp('2025-01-01')),
    'Mar 2025': (mar, pd.Timestamp('2025-03-31')),
    'Jun 2025': (jun, pd.Timestamp('2025-06-30')),
    'Sep 2025': (sep, pd.Timestamp('2025-09-30')),
    'Dec 2025': (dec, pd.Timestamp('2025-12-31')),
}

frames = []
for label, (df, snap_date) in snapshot_map.items():
    tmp = df.copy()
    tmp['SNAPSHOT']      = label
    tmp['SNAPSHOT_DATE'] = snap_date
    frames.append(tmp)

credit = pd.concat(frames, ignore_index=True)
print('Combined credit rows:', f'{len(credit):,}')
print('Columns:', credit.columns.tolist())


In [ ]:
# CUSTOMER_AGE is loan age in days, not customer age.
# Proof: correlation with days since SALE_DATE is 1.000.
dec_tmp = dec.copy()
dec_tmp['SALE_DATE_parsed'] = pd.to_datetime(dec_tmp['SALE_DATE'], errors='coerce')
dec_tmp['days_since_sale']  = (pd.Timestamp('2025-12-31') - dec_tmp['SALE_DATE_parsed']).dt.days
corr = dec_tmp[['CUSTOMER_AGE', 'days_since_sale']].corr().iloc[0, 1]
print(f'Correlation CUSTOMER_AGE vs days since sale: {corr:.4f}')
print(f'CUSTOMER_AGE range: {dec_tmp["CUSTOMER_AGE"].min()} to {dec_tmp["CUSTOMER_AGE"].max()}')
print('=> Field is mislabelled. True customer age must come from the DOB sheet.')


In [ ]:
# DOB sheet — Loan Id column has trailing space
dob = (
    dob_raw[dob_raw['Loan Id '].notna()]
    [['Loan Id ', 'date_of_birth']]
    .drop_duplicates('Loan Id ')
    .rename(columns={'Loan Id ': 'LOAN_ID', 'date_of_birth': 'DOB'})
)
dob['DOB'] = pd.to_datetime(dob['DOB'], utc=True).dt.tz_localize(None)

# Income sheet — 'Received' is total income over Duration months
inc = (
    inc_raw[inc_raw['Loan Id'].notna()]
    [['Loan Id', 'Duration', 'Received']]
    .drop_duplicates('Loan Id')
    .rename(columns={'Loan Id': 'LOAN_ID', 'Received': 'Total_Income'})
)

print(f'DOB records linked to loans:    {len(dob):,}')
print(f'Income records linked to loans: {len(inc):,}')
print(f'Dec portfolio size:             {len(dec):,}')
print(f'DOB coverage:    {len(dob)/len(dec)*100:.1f}%')
print(f'Income coverage: {len(inc)/len(dec)*100:.1f}%')


In [ ]:
def age_band(age):
    if pd.isna(age) or age < 18: return None
    if age <= 25: return '18-25'
    if age <= 35: return '26-35'
    if age <= 45: return '36-45'
    if age <= 55: return '46-55'
    return '55+'

def income_band(inc):
    if pd.isna(inc) or inc < 0: return None
    if inc < 5000:    return 'Below 5,000'
    if inc < 10000:   return '5,000-9,999'
    if inc < 20000:   return '10,000-19,999'
    if inc < 30000:   return '20,000-29,999'
    if inc < 50000:   return '30,000-49,999'
    if inc < 100000:  return '50,000-99,999'
    if inc < 150000:  return '100,000-149,999'
    return '150,000+'

credit = credit.merge(dob, on='LOAN_ID', how='left')
credit = credit.merge(inc, on='LOAN_ID', how='left')

credit['DOB'] = pd.to_datetime(credit['DOB'], errors='coerce')
credit['AGE_YEARS']          = (credit['SNAPSHOT_DATE'] - credit['DOB']).dt.days / 365.25
credit['AVG_MONTHLY_INCOME'] = credit['Total_Income'] / credit['Duration']
credit['AGE_BAND']           = credit['AGE_YEARS'].apply(age_band)
credit['INCOME_BAND']        = credit['AVG_MONTHLY_INCOME'].apply(income_band)

credit['IS_PAR30']    = (credit['DAYS_PAST_DUE'] >= 30).astype(int)
credit['IS_WRITEOFF'] = credit['ACCOUNT_STATUS_L1'].str.contains('Write Off', na=False).astype(int)
credit['IS_FPD']      = (credit['ACCOUNT_STATUS_L2'] == 'FPD').astype(int)

print('Enriched credit shape:', credit.shape)
credit.groupby('SNAPSHOT')[['LOAN_ID','IS_PAR30','IS_WRITEOFF','IS_FPD']].agg(
    Accounts=('LOAN_ID','count'),
    PAR30=('IS_PAR30','sum'),
    WriteOffs=('IS_WRITEOFF','sum'),
    FPD=('IS_FPD','sum'),
)


In [ ]:
snap_order = ['Jan 2025','Mar 2025','Jun 2025','Sep 2025','Dec 2025']

records = []
for snap in snap_order:
    df = credit[credit['SNAPSHOT'] == snap]
    n  = len(df)
    total_due  = df['TOTAL_DUE_TODAY'].sum()
    total_paid = df['TOTAL_PAID'].sum()
    records.append({
        'Snapshot'             : snap,
        'Total Accounts'       : n,
        'PAR30+ Rate (%)'      : round(df['IS_PAR30'].sum() / n * 100, 2),
        'NPL Rate (%)'         : round(df['IS_WRITEOFF'].sum() / n * 100, 2),
        'FPD Rate (%)'         : round(df['IS_FPD'].sum() / n * 100, 2),
        'Arrears Ratio (%)'    : round(df['ARREARS'].sum() / total_due * 100, 2) if total_due else 0,
        'Collection Rate (%)'  : round(total_paid / total_due * 100, 2) if total_due else 0,
        'Total Arrears (KES M)': round(df['ARREARS'].sum() / 1e6, 1),
    })

metrics = pd.DataFrame(records)
metrics


In [ ]:
snap_labels = metrics['Snapshot'].tolist()
par30      = metrics['PAR30+ Rate (%)'].tolist()
npl        = metrics['NPL Rate (%)'].tolist()
fpd        = metrics['FPD Rate (%)'].tolist()
collection = metrics['Collection Rate (%)'].tolist()
arrears_m  = metrics['Total Arrears (KES M)'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Portfolio Health — Key Metrics Across Snapshots', fontsize=14, fontweight='bold')

ax = axes[0, 0]
x = np.arange(len(snap_labels)); w = 0.38
ax.bar(x - w/2, par30, w, color=ORANGE, alpha=0.85, label='PAR30+')
ax.bar(x + w/2, npl,   w, color=NAVY,   alpha=0.85, label='NPL')
ax.set_xticks(x); ax.set_xticklabels(snap_labels, rotation=15)
ax.set_ylabel('Rate (%)'); ax.set_title('PAR30+ and NPL Rate'); ax.legend(fontsize=9)
for i, (a, b) in enumerate(zip(par30, npl)):
    ax.text(i-w/2, a+0.5, f'{a:.1f}%', ha='center', fontsize=8, color=ORANGE, fontweight='bold')
    ax.text(i+w/2, b+0.5, f'{b:.1f}%', ha='center', fontsize=8, color=NAVY,   fontweight='bold')

ax = axes[0, 1]
ax.plot(snap_labels, collection, 'o-', color=GREEN, linewidth=2.2, markersize=7)
ax.fill_between(snap_labels, collection, 65, alpha=0.1, color=GREEN)
ax.axhline(75, linestyle='--', color=AMBER, linewidth=1.2, label='75% reference')
for s, v in zip(snap_labels, collection):
    ax.annotate(f'{v:.1f}%', (s, v), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=9, color=GREEN, fontweight='bold')
ax.set_ylim(65, 82); ax.set_ylabel('Rate (%)')
ax.set_title('Collection Rate'); ax.legend(fontsize=9)

ax = axes[1, 0]
ax.plot(snap_labels, fpd, 's-', color=NAVY, linewidth=2.2, markersize=7)
ax.fill_between(snap_labels, fpd, alpha=0.1, color=NAVY)
for s, v in zip(snap_labels, fpd):
    ax.annotate(f'{v:.1f}%', (s, v), textcoords='offset points', xytext=(0, 7),
                ha='center', fontsize=9, color=NAVY, fontweight='bold')
ax.set_ylim(0, 10); ax.set_ylabel('Rate (%)')
ax.set_title('First Payment Default Rate')

ax = axes[1, 1]
bar_c = [AMBER if v < 200 else ORANGE for v in arrears_m]
bars  = ax.bar(snap_labels, arrears_m, color=bar_c, alpha=0.88, width=0.5)
for bar, v in zip(bars, arrears_m):
    ax.text(bar.get_x()+bar.get_width()/2, v+4,
            f'KES {v:.0f}M', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('KES Millions'); ax.set_title('Total Arrears Amount')

plt.tight_layout()
plt.savefig('q1_portfolio_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
age_order  = ['18-25','26-35','36-45','46-55','55+']
colors_age = [ORANGE, NAVY, GREEN, PURPLE, AMBER]

age_time = (
    credit[credit['AGE_BAND'].notna()]
    .groupby(['SNAPSHOT','AGE_BAND'])
    .agg(Total=('LOAN_ID','count'), PAR30=('IS_PAR30','sum'))
    .reset_index()
)
age_time['PAR30_rate'] = age_time['PAR30'] / age_time['Total'] * 100

pivot_age = (
    age_time
    .pivot(index='SNAPSHOT', columns='AGE_BAND', values='PAR30_rate')
    .reindex(snap_order)[age_order]
)
print('PAR30 rate (%) by age band:')
pivot_age.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

for band, col in zip(age_order, colors_age):
    vals = pivot_age[band].tolist()
    ax.plot(snap_order, vals, 'o-', color=col, linewidth=2.2, markersize=7, label=f'Age {band}')
    ax.annotate(f'{vals[-1]:.1f}%', (snap_order[-1], vals[-1]),
                textcoords='offset points', xytext=(6, 0),
                fontsize=9, color=col, fontweight='bold', va='center')

ax.plot(snap_order, par30, '--', color='black', linewidth=1.8, alpha=0.4, label='Portfolio average')
ax.set_ylim(0, 48)
ax.set_ylabel('PAR30 Rate (%)')
ax.set_title('PAR30 Rate by Age Band vs Portfolio Average')
ax.legend(fontsize=9, loc='upper left', ncol=3)

youngest_vals = pivot_age['18-25'].tolist()
ax.annotate(
    '18-25 started lowest (6.9%)\nbut climbed fastest — up 27 pts',
    xy=(snap_order[-1], youngest_vals[-1]),
    xytext=(snap_order[1], 42),
    fontsize=8.5, color=ORANGE,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF0EB', edgecolor=ORANGE),
    arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.2)
)

plt.tight_layout()
plt.savefig('q1_age_segment.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
inc_order = [
    'Below 5,000','5,000-9,999','10,000-19,999','20,000-29,999',
    '30,000-49,999','50,000-99,999','100,000-149,999','150,000+'
]

inc_metrics = (
    credit[credit['INCOME_BAND'].notna()]
    .groupby('INCOME_BAND')
    .agg(Total=('LOAN_ID','count'), PAR30=('IS_PAR30','sum'), FPD=('IS_FPD','sum'))
    .reindex(inc_order)
    .reset_index()
)
inc_metrics['PAR30_rate'] = (inc_metrics['PAR30'] / inc_metrics['Total'] * 100).round(1)
inc_metrics['FPD_rate']   = (inc_metrics['FPD']   / inc_metrics['Total'] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(inc_order)); w = 0.38
ax.bar(x - w/2, inc_metrics['PAR30_rate'], w, color=ORANGE, alpha=0.85, label='PAR30 Rate')
ax.bar(x + w/2, inc_metrics['FPD_rate'],   w, color=NAVY,   alpha=0.85, label='FPD Rate')
ax.set_xticks(x)
ax.set_xticklabels([f'KES {b}' for b in inc_order], rotation=20, ha='right', fontsize=8.5)
ax.set_ylabel('Rate (%)')
ax.set_title('PAR30 and FPD Rate by Monthly Income Band')
ax.legend(fontsize=9)
for i, (a, b) in enumerate(zip(inc_metrics['PAR30_rate'], inc_metrics['FPD_rate'])):
    ax.text(i-w/2, a+0.3, f'{a:.1f}%', ha='center', fontsize=7.5, color=ORANGE, fontweight='bold')
    ax.text(i+w/2, b+0.3, f'{b:.1f}%', ha='center', fontsize=7.5, color=NAVY,   fontweight='bold')

plt.tight_layout()
plt.savefig('q1_income_segment.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
nps = nps_raw.copy()
nps.columns = [
    'submission_id','respondent_id','submitted_at','loan_id',
    'nps_score','reason','improvement','happy_quality','happy_service',
    'payment_delay','support_difficulty','challenge_desc',
    'battery_issue','moapp_used','pref_channel','phone_locked','other_feedback'
]

# Use latest available credit status per loan
credit_latest = (
    credit[['LOAN_ID','SNAPSHOT_DATE','DAYS_PAST_DUE','ARREARS',
             'ACCOUNT_STATUS_L1','ACCOUNT_STATUS_L2']]
    .sort_values('SNAPSHOT_DATE', ascending=False)
    .drop_duplicates('LOAN_ID')
)

nps_merged = nps.merge(credit_latest, left_on='loan_id', right_on='LOAN_ID', how='left')

def nps_cat(score):
    if pd.isna(score): return None
    if score >= 9: return 'Promoter'
    if score >= 7: return 'Passive'
    return 'Detractor'

nps_merged['nps_cat'] = nps_merged['nps_score'].apply(nps_cat)

valid       = nps_merged['nps_score'].dropna()
n_total     = len(valid)
n_promoters = (valid >= 9).sum()
n_passives  = (valid.between(7, 8)).sum()
n_detractors= (valid <= 6).sum()
overall_nps = (n_promoters - n_detractors) / n_total * 100

print(f'Overall NPS score: {overall_nps:.1f}')
print(f'Promoters:   {n_promoters:,} ({n_promoters/n_total*100:.1f}%)')
print(f'Passives:    {n_passives:,}  ({n_passives/n_total*100:.1f}%)')
print(f'Detractors:  {n_detractors:,} ({n_detractors/n_total*100:.1f}%)')
print(f'Credit match rate: {nps_merged["LOAN_ID"].notna().mean()*100:.1f}%')


In [ ]:
matched = nps_merged[nps_merged['LOAN_ID'].notna()].copy()

matched['dpd_bucket'] = pd.cut(
    matched['DAYS_PAST_DUE'],
    bins=[-1, 0, 7, 30, 60, 90, 9999],
    labels=['Current (0)', '1-7 DPD', '8-30 DPD', '31-60 DPD', '61-90 DPD', '90+ DPD']
)
matched['arrears_bucket'] = pd.cut(
    matched['ARREARS'],
    bins=[-1, 0, 1000, 5000, 20000, 1e9],
    labels=['No Arrears', '< 1K', '1K-5K', '5K-20K', '20K+']
)

status_nps = (
    matched.groupby('ACCOUNT_STATUS_L2')['nps_score']
    .agg(['mean','count']).round(2)
    .sort_values('mean').reset_index()
)
dpd_nps = (
    matched.groupby('dpd_bucket', observed=True)['nps_score']
    .agg(['mean','count']).round(2).reset_index()
)
arrears_nps = (
    matched.groupby('arrears_bucket', observed=True)['nps_score']
    .agg(['mean','count']).round(2).reset_index()
)

print('NPS by account status:')
print(status_nps.to_string(index=False))
print('\nNPS by DPD bucket:')
print(dpd_nps.to_string(index=False))
print('\nNPS by arrears (KES):')
print(arrears_nps.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NPS Score vs Credit Performance', fontsize=14, fontweight='bold')

ax = axes[0]
bar_c = [RED if v < 5 else ORANGE if v < 7 else GREEN for v in status_nps['mean']]
ax.barh(status_nps['ACCOUNT_STATUS_L2'], status_nps['mean'], color=bar_c, alpha=0.85, height=0.6)
ax.axvline(7, linestyle='--', color=AMBER, linewidth=1.3, alpha=0.8)
for _, row in status_nps.iterrows():
    ax.text(row['mean']+0.05, row['ACCOUNT_STATUS_L2'],
            f"{row['mean']:.1f}", va='center', fontsize=8.5, fontweight='bold')
ax.set_xlim(0, 10); ax.set_xlabel('Avg NPS Score'); ax.set_title('By Account Status')

ax = axes[1]
dpd_c = [RED if v < 6 else ORANGE if v < 7 else GREEN for v in dpd_nps['mean']]
ax.bar(dpd_nps['dpd_bucket'].astype(str), dpd_nps['mean'], color=dpd_c, alpha=0.85, width=0.6)
ax.axhline(7, linestyle='--', color=AMBER, linewidth=1.3, alpha=0.8)
for i, row in dpd_nps.iterrows():
    ax.text(i, row['mean']+0.1, f"{row['mean']:.1f}",
            ha='center', fontsize=8.5, fontweight='bold')
ax.set_ylim(0, 9.5); ax.set_ylabel('Avg NPS Score')
ax.set_title('By Days Past Due'); ax.tick_params(axis='x', rotation=20)

ax = axes[2]
arr_c = [RED if v < 6 else ORANGE if v < 7 else GREEN for v in arrears_nps['mean']]
ax.bar(arrears_nps['arrears_bucket'].astype(str), arrears_nps['mean'],
       color=arr_c, alpha=0.85, width=0.6)
ax.axhline(7, linestyle='--', color=AMBER, linewidth=1.3, alpha=0.8)
for i, row in arrears_nps.iterrows():
    ax.text(i, row['mean']+0.1, f"{row['mean']:.2f}",
            ha='center', fontsize=8.5, fontweight='bold')
ax.set_ylim(0, 9.5); ax.set_ylabel('Avg NPS Score')
ax.set_title('By Arrears Amount (KES)'); ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('q2_nps_credit.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
tension_groups = {
    'No phone lock issue'   : nps_merged[nps_merged['phone_locked'] == 'No']['nps_score'].mean(),
    'Phone locked (paid)'   : nps_merged[nps_merged['phone_locked'] == 'Yes']['nps_score'].mean(),
    'No support difficulty' : nps_merged[nps_merged['support_difficulty'] == 'No']['nps_score'].mean(),
    'Had support difficulty': nps_merged[nps_merged['support_difficulty'] == 'Yes']['nps_score'].mean(),
    'No payment delay'      : nps_merged[nps_merged['payment_delay'] == 'No']['nps_score'].mean(),
    'Payment reflected late': nps_merged[nps_merged['payment_delay'] == 'Yes']['nps_score'].mean(),
}
labels_t = list(tension_groups.keys())
scores_t = [round(v, 2) for v in tension_groups.values()]
counts_t = [
    (nps_merged['phone_locked'] == 'No').sum(),
    (nps_merged['phone_locked'] == 'Yes').sum(),
    (nps_merged['support_difficulty'] == 'No').sum(),
    (nps_merged['support_difficulty'] == 'Yes').sum(),
    (nps_merged['payment_delay'] == 'No').sum(),
    (nps_merged['payment_delay'] == 'Yes').sum(),
]

fig, ax = plt.subplots(figsize=(12, 5))
bar_c_t = [GREEN, ORANGE, GREEN, RED, GREEN, AMBER]
bars    = ax.bar(range(len(labels_t)), scores_t, color=bar_c_t, alpha=0.88, width=0.6)
for i, (bar, v, c) in enumerate(zip(bars, scores_t, counts_t)):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.1,
            f'{v:.2f}', ha='center', fontsize=9.5, fontweight='bold')
    ax.text(bar.get_x()+bar.get_width()/2, 0.2,
            f'n={c:,}', ha='center', fontsize=8, color='white', fontweight='bold')
ax.set_xticks(range(len(labels_t)))
ax.set_xticklabels(labels_t, rotation=15, ha='right', fontsize=9)
ax.axhline(7, linestyle='--', color=AMBER, linewidth=1.3, alpha=0.7)
for pair, col in [((0,1), ORANGE), ((2,3), RED), ((4,5), AMBER)]:
    drop  = scores_t[pair[0]] - scores_t[pair[1]]
    mid   = (pair[0] + pair[1]) / 2
    y_top = max(scores_t[pair[0]], scores_t[pair[1]]) + 0.5
    ax.annotate('', xy=(pair[1], y_top+0.1), xytext=(pair[0], y_top+0.1),
                arrowprops=dict(arrowstyle='<->', color=col, lw=1.6))
    ax.text(mid, y_top+0.35, f'-{drop:.2f} pts',
            ha='center', fontsize=9, color=col, fontweight='bold')
for sep in [1.5, 3.5]:
    ax.axvline(sep, color='#DDE3EC', linewidth=1.5, linestyle=':')
ax.set_ylim(0, 10.5); ax.set_ylabel('Avg NPS Score')
ax.set_title('Collections Friction vs Customer Satisfaction')
plt.tight_layout()
plt.savefig('q2_tension.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
cr_dec = credit[credit['SNAPSHOT'] == 'Dec 2025'].copy()

# CUSTOMER_AGE vs days since sale
cr_dec['SALE_DATE_parsed'] = pd.to_datetime(cr_dec['SALE_DATE'], errors='coerce')
cr_dec['days_since_sale']  = (pd.Timestamp('2025-12-31') - cr_dec['SALE_DATE_parsed']).dt.days
corr = cr_dec[['CUSTOMER_AGE','days_since_sale']].corr().iloc[0, 1]

# Linkage coverage
dec_ids = set(cr_dec['LOAN_ID'])
coverage = pd.DataFrame({
    'Table'        : ['DOB', 'Income', 'Gender', 'NPS Survey'],
    'Matched'      : [len(dec_ids & set(dob['LOAN_ID'])),
                      len(dec_ids & set(inc['LOAN_ID'])),
                      round(len(dec_ids) * 0.506),
                      len(set(nps['loan_id'].dropna()) & dec_ids)],
    'Portfolio'    : [len(dec_ids)] * 4,
})
coverage['Coverage (%)'] = (coverage['Matched'] / coverage['Portfolio'] * 100).round(1)

# Field issues
diff_balance = (cr_dec['BALANCE'] != cr_dec['CLOSING_BALANCE']).sum()
diff_paid    = (cr_dec['TOTAL_PAID'] != cr_dec['TOTAL_PAID_WITH_ADJUSTMENTS_15D']).sum()
null_payment = cr_dec['PAYMENT_AMOUNT'].isna().sum()
par30_wo     = ((cr_dec['ACCOUNT_STATUS_L2']=='PAR 30') &
                 cr_dec['ACCOUNT_STATUS_L1'].str.contains('Write Off', na=False)).sum()

print(f'CUSTOMER_AGE vs days since sale — correlation: {corr:.4f}')
print(f'=> Field is mislabelled. Rename to LOAN_AGE_DAYS.\n')
print('Demographic linkage coverage (Dec portfolio):')
print(coverage.to_string(index=False))
print(f'\nBALANCE != CLOSING_BALANCE: {diff_balance} rows (redundant field)')
print(f'PAYMENT_AMOUNT null: {null_payment:,} / {len(cr_dec):,} rows ({null_payment/len(cr_dec)*100:.1f}%)')
print(f'TOTAL_PAID gap rows: {diff_paid:,} (ADJUSTMENT_AMOUNT 99.9% null — unexplained)')
print(f'PAR 30 (L2) containing Write Off (L1): {par30_wo:,} rows — status hierarchy broken')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Data Quality Audit', fontsize=14, fontweight='bold')

ax = axes[0]
tables  = coverage['Table'].tolist()
matched_pct = coverage['Coverage (%)'].tolist()
missing_pct = [100 - m for m in matched_pct]
x = np.arange(len(tables))
ax.bar(x, matched_pct, color=GREEN, alpha=0.85, width=0.55, label='Matched')
ax.bar(x, missing_pct, bottom=matched_pct, color=RED, alpha=0.5, width=0.55, label='Not matched')
for i, (m, mis) in enumerate(zip(matched_pct, missing_pct)):
    ax.text(i, m/2,      f'{m:.1f}%',   ha='center', va='center', fontsize=9.5, fontweight='bold', color='white')
    ax.text(i, m+mis/2,  f'{mis:.1f}%', ha='center', va='center', fontsize=9.5, fontweight='bold', color='white')
ax.set_xticks(x); ax.set_xticklabels(tables)
ax.set_ylabel('% of Dec Portfolio'); ax.set_title('Demographic Linkage Coverage')
ax.legend(fontsize=9)

ax = axes[1]
issue_labels = [
    'PAYMENT_AMOUNT\n(99.9% null)',
    'ADJUSTMENT_AMOUNT\n(99.9% null)',
    'TOTAL_PAID gap\n(1,896 rows)',
    'PAR30 contains\nWrite Off accounts',
    'Unnamed empty\ncolumn',
]
severity   = [3, 3, 2, 3, 1]
sev_colors = [RED if s==3 else AMBER if s==2 else MUTED for s in severity]
ax.barh(issue_labels, severity, color=sev_colors, alpha=0.85, height=0.55)
ax.set_xlim(0, 4)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Minor', 'Moderate', 'Major'], fontsize=9)
ax.set_title('Schema Issues by Severity')
patches = [mpatches.Patch(color=RED, alpha=0.85, label='Major'),
           mpatches.Patch(color=AMBER, alpha=0.85, label='Moderate'),
           mpatches.Patch(color=MUTED, alpha=0.85, label='Minor')]
ax.legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig('q3_data_quality.png', dpi=150, bbox_inches='tight')
plt.show()
